We use three methods that are suitable for categorical survey data:

| Step | Method | Purpose |
|---|---|---|
| 1 | **Frequency analysis** | Find very rare categories |
| 2 | **Domain / rule validation** | Find invalid or impossible responses |
| 3 | **Consistency checking** | Find contradictory / inconsistent responses |

In [1]:
import pandas as pd
df = pd.read_pickle("dataset/dataset_v2.pkl")
df.dtypes

A1_age_group                      category
A2_gender                         category
A3_religion                       category
A4_marital_status                 category
A5_residence                      category
A6_occupation                     category
B1_family_type                    category
B2_family_time_group              category
B3_family_relationship            category
B4_in_person_feeling              category
C1_height_group                   category
C2_weight_group                   category
C3_sitting_duration               category
C4_physical_problems              category
D1_years_group                    category
D2_daily_watch_time               category
D3_peak_watch_time                category
D4_multitasking_while_watching    category
E1_sleep_hours                    category
E2_sleep_trouble                  category
E3_exercise_involvement           category
E4_substance_use                  category
F1_attention_decreased            category
F2_memory_r

---
## Frequency Analysis

**Goal:** Find categories that very few respondents selected.

**Why:** In categorical data, an *outlier* is a category that appears very rarely. Rare categories:
- make statistical tests (e.g. chi-square) unreliable because of very small cell counts,
- can produce misleading percentages in crosstabs.

In [2]:
threshold = 0.01  # 1% of rows

rare_rows = []
for col in df.columns:
    counts = df[col].value_counts(dropna=False)
    percent = counts / len(df)
    for cat in percent[percent < threshold].index:
        rare_rows.append({
            "column": col,
            "category": cat,
            "count": counts[cat],
            "percent": round(percent[cat] * 100, 2),
        })

rare_df = pd.DataFrame(rare_rows)
print(f"Total rows: {len(df)}")
print(f"Columns with rare categories (< {threshold:.0%}): {rare_df['column'].nunique()}")
rare_df

Total rows: 506
Columns with rare categories (< 1%): 3


,column,category,count,percent
0,A4_marital_status,Divorced,5,0.99
1,B1_family_type,Joint Extended Family,4,0.79
2,B3_family_relationship,Bad,4,0.79
3,B3_family_relationship,Very Bad,1,0.20


In [3]:
# Full frequency table of the flagged columns
for col in rare_df["column"].unique():
    freq = df[col].value_counts().to_frame("count")
    freq["percent"] = (freq["count"] / len(df) * 100).round(2)
    print(f"\n{col}")
    print(freq)


A4_marital_status
                   count  percent
A4_marital_status                
Unmarried            327    64.62
Married              174    34.39
Divorced               5     0.99

B1_family_type
                       count  percent
B1_family_type                       
Single family            313    61.86
Joint Family             183    36.17
Extended Family            6     1.19
Joint Extended Family      4     0.79

B3_family_relationship
                        count  percent
B3_family_relationship                
Very Good                 322    63.64
Good                      131    25.89
Moderate                   48     9.49
Bad                         4     0.79
Very Bad                    1     0.20


---
## Domain / Rule Validation

**Goal:** Find values that are invalid or impossible.

**Why:** Survey data can contain typos, extra spaces, missing answers, or values that were never options in the questionnaire. These are true errors and must be fixed or removed.

**Method:**
1. Define the **allowed answers** for every question (based on the questionnaire).
2. Check every column for values outside that allowed list.
3. Check for **missing values**.
4. Check for **duplicate rows** (the same response submitted twice).

In [4]:
allowed = {
    "A1_age_group": ["16-20", "21-25", "26-35"],
    "A2_gender": ["Male", "Female"],
    "A3_religion": ["Islam", "Hindu/Christian"],
    "A4_marital_status": ["Unmarried", "Married", "Divorced"],
    "A5_residence": ["Urban Area", "Rural Area"],
    "A6_occupation": ["Student", "Service Holder", "Businessperson", "Others"],
    "B1_family_type": ["Single family", "Joint Family", "Extended Family", "Joint Extended Family"],
    "B2_family_time_group": ["Up to 1 hour", "1-3 hours", "3-5 hours", "5-8 hours", "More than 8 hours"],
    "B3_family_relationship": ["Very Good", "Good", "Moderate", "Bad", "Very Bad"],
    "B4_in_person_feeling": ["Very Comfortable", "Somewhat Comfortable", "Neutral", "Somewhat Uncomfortable", "Very Uncomfortable"],
    "C1_height_group": ["Up to 160 cm", "160-165 cm", "165-170 cm", "170-175 cm", "Above 175 cm"],
    "C2_weight_group": ["Up to 50 kg", "50-60 kg", "60-70 kg", "70-80 kg", "Above 80 kg"],
    "C3_sitting_duration": ["Less than 30 minutes", "30 minutes–1 hour", "1–2 hours", "More than 2 hours"],
    "C4_physical_problems": ["Never", "Rarely", "Sometimes", "Often"],
    "D1_years_group": ["Up to 1 year", "1-2 years", "2-3 years", "3-5 years", "More than 5 years"],
    "D2_daily_watch_time": ["Less than 1 hour", "1 hour", "2 hours", "3 hours", "More than that"],
    "D3_peak_watch_time": ["Morning", "Afternoon", "Before Study", "During Work Break", "Leisure Time", "Before Sleeping"],
    "D4_multitasking_while_watching": ["No", "Sometimes", "Yes, frequently"],
    "E1_sleep_hours": ["Less than 6 hours", "6–8 hours", "More than 8 hours"],
    "E2_sleep_trouble": ["No", "Sometimes", "Yes"],
    "E3_exercise_involvement": ["Never", "Rarely", "Sometimes", "Regularly"],
    "E4_substance_use": ["No", "Yes", "Sometimes", "Used to do before", "Prefer not to say"],
    "F1_attention_decreased": ["Did not decrease at all", "Did not decrease", "Decreased a little", "Decreased a lot"],
    "F2_memory_reduced": ["Did not decrease at all", "Did not decrease", "Decreased a little", "Decreased a lot"],
    "F3_learning_interest_decreased": ["Did not decrease at all", "Did not decrease", "Decreased a little", "Decreased a lot"],
    "F4_performance_affected": ["Very negative", "Somewhat negative", "No effect", "Somewhat positive", "Very positive"],
    "G1_mood_swings": ["Never", "Sometimes", "Often", "Always"],
    "G2_fomo_anxiety": ["Never", "Sometimes", "Often", "Always"],
    "G3_social_comparison": ["No", "Sometimes", "Yes, frequently"],
    "G4_social_isolation": ["No", "Sometimes", "Yes, frequently"],
    "G5_mental_stress": ["No", "Not Sure", "Yes"],
    "G6_panic_attacks": ["Never", "Very Rarely", "Sometimes", "Yes, regularly"],
    "G7_behavior_type": ["Introvert", "Ambivert", "Extrovert"],
    "G8_addictive_watching": ["Never", "Very Rarely", "Sometimes", "Often", "Always"],
    "H1_mental_state_change": ["No Change", "Some Change", "Major Change", "Not Sure"],
    "H2_content_type_effect": ["Positive Impact", "No Impact", "Negative Impact", "Not Sure"],
    "H3_distraction_after_viral": ["Never", "Rarely", "Sometimes", "Often"],
    "H4_brain_pressure": ["No Stress", "Low Stress", "Moderate Stress", "High Stress"],
    "H5_sleep_quality_change": ["Sleep Quality Unchanged", "Slight Change in Sleep Quality", "Poor Sleep Quality", "Not Sure"],
    "H6_emotional_reels_focus_loss": ["Do Not Lose Focus", "Lose Focus Rarely", "Lose Focus Sometimes", "Lose Focus Frequently"],
    "H7_curiosity_driven_rewatch": ["Not Interested", "Rarely Interested", "Sometimes Interested", "More Interest in Watching Reels"],
}

# 1. Every column must have a rule
missing_rules = set(df.columns) - set(allowed)
print("Columns without a rule:", missing_rules or "None")

# 2. Values outside the allowed list
invalid_rows = []
for col, values in allowed.items():
    bad = df.loc[df[col].notna() & ~df[col].isin(values), col]
    for idx, val in bad.items():
        invalid_rows.append({"row": idx, "column": col, "invalid_value": val})

invalid_df = pd.DataFrame(invalid_rows, columns=["row", "column", "invalid_value"])
print("Invalid values found:", len(invalid_df))
invalid_df

Columns without a rule: None
Invalid values found: 0


,row,column,invalid_value


In [5]:
# 3. Missing values
missing = df.isna().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "None")

# 4. Duplicate rows
duplicates = df[df.duplicated(keep=False)]
print("\nDuplicate rows (extra copies):", df.duplicated().sum())
duplicates

Missing values per column:
None

Duplicate rows (extra copies): 1


,A1_age_group,A2_gender,A3_religion,A4_marital_status,A5_residence,A6_occupation,B1_family_type,B2_family_time_group,B3_family_relationship,B4_in_person_feeling,...,G6_panic_attacks,G7_behavior_type,G8_addictive_watching,H1_mental_state_change,H2_content_type_effect,H3_distraction_after_viral,H4_brain_pressure,H5_sleep_quality_change,H6_emotional_reels_focus_loss,H7_curiosity_driven_rewatch
65,21-25,Male,Islam,Unmarried,Urban Area,Student,Single family,1-3 hours,Very Good,Very Comfortable,...,Never,Ambivert,Never,Some Change,Not Sure,Sometimes,No Stress,Slight Change in Sleep Quality,Lose Focus Sometimes,Sometimes Interested
128,21-25,Male,Islam,Unmarried,Urban Area,Student,Single family,1-3 hours,Very Good,Very Comfortable,...,Never,Ambivert,Never,Some Change,Not Sure,Sometimes,No Stress,Slight Change in Sleep Quality,Lose Focus Sometimes,Sometimes Interested


---
## Consistency Checking

**Goal:** Find respondents whose answers **contradict each other**.

**Why:** A single answer can be valid on its own, but impossible or illogical when combined with another answer. This usually means the respondent answered carelessly, or misunderstood a question.

**Method:** Define logical rules between related questions. Each rule describes a contradictory combination of answers:

| Rule | Contradiction |
|---|---|
| R1 | Age 16-20 but Married/Divorced |
| R2 | Watches less than 1 hour a day but *always* feels addicted |
| R3 | Watches more than 3 hours a day but *never* feels addicted |
| R4 | Says *no* mental stress but reports *high* brain pressure |
| R5 | Has panic attacks *regularly* but says *no* mental stress |
| R6 | Attention *decreased a lot* but *does not lose focus* |
| R7 | *No* sleep trouble but *poor* sleep quality |

A respondent who breaks **one** rule may just be an honest edge case. A respondent who breaks **two or more** rules is a strong candidate for an inconsistent (careless) response.

In [6]:
rules = {
    "R1_teen_married": (df["A1_age_group"] == "16-20")
        & df["A4_marital_status"].isin(["Married", "Divorced"]),
    "R2_low_watch_always_addicted": (df["D2_daily_watch_time"] == "Less than 1 hour")
        & (df["G8_addictive_watching"] == "Always"),
    "R3_high_watch_never_addicted": (df["D2_daily_watch_time"] == "More than that")
        & (df["G8_addictive_watching"] == "Never"),
    "R4_no_stress_high_pressure": (df["G5_mental_stress"] == "No")
        & (df["H4_brain_pressure"] == "High Stress"),
    "R5_regular_panic_no_stress": (df["G6_panic_attacks"] == "Yes, regularly")
        & (df["G5_mental_stress"] == "No"),
    "R6_attention_drop_no_focus_loss": (df["F1_attention_decreased"] == "Decreased a lot")
        & (df["H6_emotional_reels_focus_loss"] == "Do Not Lose Focus"),
    "R7_no_sleep_trouble_poor_quality": (df["E2_sleep_trouble"] == "No")
        & (df["H5_sleep_quality_change"] == "Poor Sleep Quality"),
}

rule_df = pd.DataFrame(rules)

summary = rule_df.sum().to_frame("violations")
summary["percent"] = (summary["violations"] / len(df) * 100).round(2)
summary

,violations,percent
R1_teen_married,2,0.40
R2_low_watch_always_addicted,4,0.79
R3_high_watch_never_addicted,20,3.95
R4_no_stress_high_pressure,7,1.38
R5_regular_panic_no_stress,12,2.37
R6_attention_drop_no_focus_loss,18,3.56
R7_no_sleep_trouble_poor_quality,15,2.96


In [7]:
# Number of rules each respondent breaks
df_check = df.copy()
df_check["rule_violations"] = rule_df.sum(axis=1)

print("Respondents by number of rules broken:")
print(df_check["rule_violations"].value_counts().sort_index())

# Strong candidates: 2 or more contradictions
inconsistent = df_check[df_check["rule_violations"] >= 2]
print(f"\nStrongly inconsistent respondents (2+ rules): {len(inconsistent)}")
# Which rules each of them breaks
flags = rule_df.loc[inconsistent.index]
flags.loc[:, flags.any()].assign(rule_violations=inconsistent["rule_violations"])

Respondents by number of rules broken:
rule_violations
0    435
1     66
2      3
3      2
Name: count, dtype: int64

Strongly inconsistent respondents (2+ rules): 5


,R1_teen_married,R2_low_watch_always_addicted,R3_high_watch_never_addicted,R4_no_stress_high_pressure,R5_regular_panic_no_stress,R6_attention_drop_no_focus_loss,R7_no_sleep_trouble_poor_quality,rule_violations
98,False,False,False,True,True,False,True,3
286,True,True,False,False,False,False,True,3
289,False,False,True,False,False,True,False,2
317,False,False,True,False,False,True,False,2
427,False,True,False,False,False,False,True,2


**Interpretation:**
- **1 rule broken:** keep the respondent. The combination is unusual but possible (e.g. someone may sleep without trouble but still feel their sleep quality got worse).
- **2 rules broken:** keep the respondent, but note them as borderline cases.
- **3+ rules broken:** remove. They are the strongest candidates for careless or inconsistent answers.

---
## Summary

| Step | Method | Finding | Action |
|---|---|---|---|
| 1 | Frequency analysis | Rare categories in a few columns | **Merge** into nearby categories |
| 2 | Domain / rule validation | Invalid values, missing values, duplicates | **Fix or remove** errors and duplicates |
| 3 | Consistency checking | Respondents with contradictory answers | **Keep** those breaking 1-2 rules; **remove** those breaking 3+ |

# Fix the Issues

In [8]:
# Merge "Very Bad" into "Bad" (B3_family_relationship)
col = "B3_family_relationship"
old = df[col]
new_categories = [c for c in old.cat.categories if c != "Very Bad"]
df[col] = pd.Categorical(
    old.astype(str).replace({"Very Bad": "Bad"}),
    categories=new_categories,
    ordered=old.cat.ordered,
)

freq = df[col].value_counts().to_frame("count")
freq["percent"] = (freq["count"] / len(df) * 100).round(2)
print(freq)

                        count  percent
B3_family_relationship                
Very Good                 322    63.64
Good                      131    25.89
Moderate                   48     9.49
Bad                         5     0.99


In [9]:
# Remove respondents who break 3+ consistency rules
# (respondents breaking 1-2 rules are kept)
# Uses rule_df computed earlier, so this must run before the index is reset
to_drop = rule_df.index[rule_df.sum(axis=1) >= 3]
before = len(df)
df = df.drop(index=to_drop)
print(f"Dropped rows: {list(to_drop)}")
print(f"Rows before: {before} | after: {len(df)} | removed: {before - len(df)}")

Dropped rows: [98, 286]
Rows before: 506 | after: 504 | removed: 2


In [10]:
# Remove duplicate rows (keep the first copy)
before = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
print(f"Rows before: {before} | after: {len(df)} | removed: {before - len(df)}")

Rows before: 504 | after: 503 | removed: 1


In [11]:
df.shape

(503, 41)